
# DP-ERM Programming Assignment — Output vs Objective vs Gradient Perturbation

**Goal:** Empirically show how strong convexity (lambda), dimension (d), and dataset size (n) affect the privacy–utility tradeoff.  
**Mechanisms:** Output perturbation, Objective perturbation, Gradient perturbation (full-batch DP-GD).

# Dataset loader and important ML functions (Do Not Edit)

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.model_selection import train_test_split


sns.set_theme(style="whitegrid")
sns.set_palette("tab10")
def seed_all(seed=42):
    import random, os
    random.seed(seed); np.random.seed(seed)
seed_all(7)

def load_dataset(which="breast_cancer", n=None, d=None, test_size=0.3, seed=0):
    rng = np.random.RandomState(seed)
    if which == "breast_cancer":
        data = load_breast_cancer()
        X = data.data.astype(float)
        y = data.target.astype(int)
        y = np.where(y==1, 1, -1)
    else:
        n_total = 600 if n is None else max(n, 20)
        d_raw = 40 if d is None else max(d, 40)
        X = rng.normal(size=(n_total, d_raw))
        wtrue = rng.normal(size=d_raw)
        wtrue /= (np.linalg.norm(wtrue)+1e-12)
        logits = X @ wtrue #+ 0.25 * rng.normal(size=n_total)
        y = np.where(logits >= 0, 1, -1)

    X = (X - X.mean(axis=0, keepdims=True)) / (X.std(axis=0, keepdims=True)+1e-12)

    if d is not None and d < X.shape[1]:
        U,S,Vt = np.linalg.svd(X, full_matrices=False)
        X = X @ Vt[:d].T

    if n is not None and n < X.shape[0]:
        idx = rng.permutation(X.shape[0])[:n]
        X = X[idx]
        y = y[idx]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

    def clip_rows(X, clip_norm=1.0):
        norms = np.linalg.norm(X, axis=1, keepdims=True)+1e-12
        scale = np.minimum(1.0, clip_norm/norms)
        return X*scale

    X_train = clip_rows(X_train, 1.0)
    X_test  = clip_rows(X_test, 1.0)
    return X_train, y_train, X_test, y_test

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    neg = ~pos
    # for z >= 0: 1/(1+exp(-z)) is safe (exp(-z) <= 1)
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    # for z < 0: rewrite sigmoid(z) = exp(z)/(1+exp(z)) to avoid exp(-z) overflow
    ez = np.exp(z[neg])                  # ez <= 1 here
    out[neg] = ez / (1.0 + ez)
    return out

def logistic_grad(w, X, y):
    # g_i = -(y_i x_i) / (1 + exp(y_i w^T x_i)) = -(y_i x_i) * sigmoid(-y_i w^T x_i)
    z = y * (X @ w)                      # can be large in magnitude
    s = np.empty_like(z)
    pos = z >= 0
    # sigmoid(-z) stably
    # if z >= 0: sigmoid(-z) = exp(-z)/(1+exp(-z)) with exp(-z) <= 1 (safe)
    s[pos] = np.exp(-z[pos]) / (1.0 + np.exp(-z[pos]))
    # if z < 0: sigmoid(-z) = 1/(1+exp(z)) with exp(z) <= 1 (safe)
    ez = np.exp(z[~pos])
    s[~pos] = 1.0 / (1.0 + ez)
    g = - (y[:, None] * X) * s[:, None]
    return g.mean(axis=0)

def risk(w, X, y, lam):
    p = sigmoid(X@w)
    y01 = (y>0).astype(float)
    loss = log_loss(y01, p, labels=[0,1])
    return loss

def train_baseline_gd(X, y, lam, steps=2000, lr=0.1, seed=0):
    w = np.zeros(X.shape[1])
    for _ in range(steps):
        g = logistic_grad(w, X, y) + lam*w
        w -= lr*g
    return w


# Utilities for metrics (Do not edit)

In [20]:
from dataclasses import dataclass
from typing import Dict, Any, List

@dataclass
class MetricSummary:
    mean: float
    lower: float
    upper: float
    std: float
    n: int

def ci_mean(x, alpha=0.05):
    """
    Utility to compute mean and confidence intervals
    x: array-like of values
    alpha: significance level (default 0.05 for 95% CI)
    """
    x = np.array(x, dtype=float)
    m = x.mean()
    s = x.std(ddof=1) if x.size>1 else 0.0
    from math import sqrt
    try:
        from statistics import NormalDist
        z = NormalDist().inv_cdf(1-alpha/2)
    except Exception:
        z = 1.96 if abs(alpha-0.05) < 1e-6 else 1.96
    half = z * s/np.sqrt(x.size) if x.size>1 else 0.0
    return MetricSummary(mean=m, lower=m-half, upper=m+half, std=s, n=x.size)

def aggregate_seed_results(df: pd.DataFrame, group_cols: List[str], metric: str, alpha=0.05):
    """
    Utility to aggregate results over random seeds.
    df: dataframe with results
    group_cols: columns to group by (e.g. method, eps, lam)
    metric: metric column to aggregate (e.g. excess_risk, test_acc)
    alpha: significance level for confidence intervals (default 0.05 for 95% CI)
    """
    rows = []
    if df.empty:
        return pd.DataFrame(columns=group_cols + [f"{metric}_mean", f"{metric}_lo", f"{metric}_hi", f"{metric}_std", "num_seeds"])
    for keys, sub in df.groupby(group_cols):
        stats = ci_mean(sub[metric].values, alpha=alpha)
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: val for col,val in zip(group_cols, keys)}
        row.update({f"{metric}_mean": stats.mean,
                    f"{metric}_lo": stats.lower,
                    f"{metric}_hi": stats.upper,
                    f"{metric}_std": stats.std,
                    "num_seeds": stats.n})
        rows.append(row)
    return pd.DataFrame(rows)

## Output Perturbation (Q1)

In [21]:
# ----- Output Perturbation (DONE) -----
def output_perturbation(w_hat, n, lam, eps, delta, L=1.0, rng=None):
    """
    Output-perturbation for strongly-convex ERM (pure epsilon-DP).
    Assumptions (as in lecture):
      - per-example loss is L-Lipschitz in w (we set L=1 with row clipping),
      - objective is lambda-strongly convex via L2 regularization,
      - convex set is R^d (projection = identity).

    We add spherical-Laplace noise b with density proportional to
        exp( - (n * lam * eps / (2L)) * ||b||_2 )
    and return w_hat + b.

    Parameters:
      w_hat : np.ndarray, ERM minimizer (baseline GD solution).
      n     : int, number of samples
      lam   : float, L2 regularization (strong convexity parameter)
      eps   : float, epsilon (delta is unused here; pure DP)
      delta : float, unused (kept for interface consistency)
      L     : float, Lipschitz constant for per-example loss (default 1.0)
      rng   : np.random.RandomState

    Returns:
      w_priv : np.ndarray
    """
    if rng is None:
        rng = np.random.RandomState()

    d = int(w_hat.shape[0])
    # Parameter of spherical-Laplace: density ∝ exp(-eps_alpha * ||x||_2)
    eps_alpha = (n * max(lam, 1e-12) * max(eps, 1e-12)) / (2.0 * max(L, 1e-12))

    # Sample direction uniformly on the sphere
    u = rng.normal(size=d)
    u /= (np.linalg.norm(u) + 1e-12)

    # Sample radius R ~ Gamma(shape=d, scale=1/eps_alpha)
    R = rng.gamma(shape=d, scale=1.0 / max(eps_alpha, 1e-18))

    b = R * u
    return w_hat + b


## Objective Perturbation (Q2)

In [22]:
# ----- Objective Perturbation (TODO) -----
def sample_spherical_laplace(eps_alpha, d, rng):
    """
    Sample from spherical Laplace distribution in d dimensions with
    parameter eps_alpha.
    Output: vector of shape (d,) sampled from exp(-eps_alpha*||x||_2)
    """
    u = rng.normal(size=d); u /= (np.linalg.norm(u)+1e-12)
    r = rng.gamma(shape=d, scale=1.0/eps_alpha)
    return r*u

# ----- Objective Perturbation (DONE) -----
def train_objective_perturbed(X, y, lam, eps, L=1.0, steps=5000, lr=0.01, seed=0):
    """
    Objective perturbation for logistic regression (pure epsilon-DP), matching Lecture 3.2.
    Solve: (1/n) * sum ell(w;x_i,y_i) + (lam/2)||w||^2 + (1/n) <b, w>,
           with b ~ spherical-Laplace(beta), beta = eps/(2L).  (NO factor n)
    Assumes per-example gradient bound L (here L=1 via clipping).
    Requires lam > c, where for clipped logistic c <= 1/4.
    """
    rng = np.random.RandomState(seed)
    n, d = X.shape

    c_bound = 0.25
    if lam <= c_bound:
        raise ValueError(f"Objective perturbation requires lam > c (c≈{c_bound}); got lam={lam}")

    beta = (max(eps, 1e-12)) / (2.0 * max(L, 1e-12))
    u = rng.normal(size=d); u /= (np.linalg.norm(u) + 1e-12)
    r = rng.gamma(shape=d, scale=1.0 / beta)
    b = r * u

    w = np.zeros(d, dtype=float)
    for _ in range(steps):
        g = logistic_grad(w, X, y) + lam * w + (b / n)  # keep 1/n here
        w -= lr * g
    return w



## Gradient Perturbation (Q3)

In [23]:
# ----- Gradient Perturbation (Full-batch DP-GD; TODO) -----
def zcdp_sigma_for_gd(eps, delta, T, Delta):
    """
    Use this function to compute the Gaussian noise sigma to use in each iteration of DP-GD
    to ensure (eps, delta)-DP after T steps.
    This uses zCDP composition.
    eps, delta: approx DP privacy parameters
    T: number of steps
    Delta: sensitivity of the average gradient
    """
    a = np.log(1.0/delta)
    t = -np.sqrt(a) + np.sqrt(a+eps)
    rho = max(1e-18, t**2)
    return Delta*np.sqrt(T/(2.0*rho))

# ----- Gradient Perturbation (Full-batch DP-GD; DONE) -----
def project_l2_ball(w, radius):
    if radius is None:  # no projection
        return w
    nrm = np.linalg.norm(w)
    if nrm <= radius:
        return w
    return w * (radius / (nrm + 1e-12))

def train_gradient_perturbed_gd(X, y, lam, eps, delta, steps=200, lr=0.1,
                                L=1.0, seed=0, proj_radius=None, average=True):
    rng = np.random.RandomState(seed)
    n, d = X.shape
    Delta = (2.0 * max(L, 1e-12)) / max(n, 1)
    sigma = zcdp_sigma_for_gd(eps=eps, delta=delta, T=steps, Delta=Delta)

    w = np.zeros(d, dtype=float)
    w_sum = np.zeros(d, dtype=float)
    for _ in range(steps):
        g = logistic_grad(w, X, y) + lam * w
        noise = rng.normal(loc=0.0, scale=sigma, size=d)
        w = w - lr * (g + noise)
        w = project_l2_ball(w, proj_radius)
        if average:
            w_sum += w

    w_out = (w_sum / steps) if average else w
    return w_out, float(sigma)


## Experiment & Plotting Scaffolding (Q4, set Liptschitz_L)

* Add the right Lipschitz parameter here and argue in the hand-in why that is correct
* If you want to pass different learning rate, or compute additional metrics for each run, edit them here

In [24]:
def run_all_mechanisms(X_train, y_train, X_test, y_test, lam, eps, delta,
                       steps_dpgd=2500, steps_output_gd=2500, steps_objective_gd=2500,
                       seed=0, safe_objective=True, c_bound=0.25):
    """
    Run baseline + mechanisms and return dict of results.
    If safe_objective=True and lam <= c_bound, skip objective perturbation
    (so sweeps over tiny lambdas don't crash when we aren't plotting objective).
    """
    LIPSCHITZ_L = 1.0  # row clipping enforces this

    w0 = train_baseline_gd(X_train, y_train, lam=lam, steps=steps_output_gd, lr=0.1, seed=seed)
    base_risk = risk(w0, X_train, y_train, lam)

    results = {
        "baseline": {"w": w0, "train_risk": base_risk}
    }

    # Output perturbation (pure epsilon-DP)
    w_out = output_perturbation(
        w0.copy(), n=X_train.shape[0], lam=lam, eps=eps, delta=delta,
        L=LIPSCHITZ_L, rng=np.random.RandomState(seed+1)
    )
    results["output"] = {"w": w_out}

    # Objective perturbation (pure epsilon-DP) — skip if lambda <= c
    if safe_objective and lam <= c_bound:
        results["objective"] = {"w": None, "skipped": True, "reason": f"lam <= c_bound ({c_bound})"}
    else:
        try:
            w_obj = train_objective_perturbed(
                X_train, y_train, lam=lam, eps=eps, L=LIPSCHITZ_L,
                steps=steps_objective_gd, lr=0.1, seed=seed+2
            )
            results["objective"] = {"w": w_obj}
        except ValueError as e:
            # Be robust even if called elsewhere with bad lambda
            results["objective"] = {"w": None, "error": str(e)}

    # Gradient perturbation ((eps,delta)-DP)
    w_gd, sigma_g = train_gradient_perturbed_gd(
        X_train, y_train, lam=lam, eps=eps, delta=delta,
        steps=steps_dpgd, lr=0.2, seed=seed+3
    )
    results["grad"] = {"w": w_gd, "sigma_g": sigma_g}

    return results

def summarize(results, X_train, y_train, lam):
    """
    Compute excess empirical risk for any mechanisms present in `results`.
    Skips ones with missing/invalid weights (e.g., objective skipped for small lambda).
    """
    R0 = results["baseline"]["train_risk"]
    out = {}
    for k in ("output", "objective", "grad"):
        wk = results.get(k, {}).get("w", None)
        if wk is None:
            continue
        # guard against NaNs if any
        if hasattr(wk, "__len__") and not np.all(np.isfinite(wk)):
            continue
        Rk = risk(wk, X_train, y_train, lam)
        out[k] = dict(excess_risk=float(abs(Rk - R0)))
    return out


# Code for sweeping

* In the following cell, there is a dummy code for sweeping over some epsilon and some lambda.
* You will have to add more functions here for sweeping over other parameters that you want to plot with respect to.
* All sweeping codes should be added in this cell

In [25]:
def sweep_eps_for_lams(X_train, y_train, X_test, y_test, lams, eps_list, delta, steps_dpgd=250, steps_output_gd=250, steps_objective_gd=250, seed=0, mech="output", seeds=None):
    """Return long-form DataFrame across lambdas, epsilons, and seeds."""
    records = []
    if seeds is None:
        seeds = [seed]
    for lam in lams:
        for eps in eps_list:
            for s in seeds:
                R = run_all_mechanisms(X_train, y_train, X_test, y_test, lam=lam, eps=eps, delta=delta, steps_dpgd=steps_dpgd, steps_output_gd=steps_output_gd, steps_objective_gd=steps_objective_gd,  seed=s)
                S = summarize(R, X_train, y_train, lam=lam)
                rec = {"lambda": lam, "epsilon": eps, "seed": s}
                rec.update(S[mech])
                records.append(rec)
    return pd.DataFrame(records)

# --------------------- Additional sweeps for Q5 ---------------------

def sweep_lambda_for_eps(X_train, y_train, X_test, y_test, lam_list, eps_list, delta,
                         steps_dpgd=1000, steps_output_gd=2000, steps_objective_gd=2000,
                         seeds=None, mech="output"):
    """
    For each epsilon in eps_list, sweep lambda over lam_list and aggregate excess risk.
    Returns a long-form DataFrame with columns: epsilon, lambda, excess_risk, seed.
    """
    records = []
    if seeds is None:
        seeds = [0]
    for eps in eps_list:
        for lam in lam_list:
            for s in seeds:
                R = run_all_mechanisms(X_train, y_train, X_test, y_test, lam=lam, eps=eps, delta=delta,
                                       steps_dpgd=steps_dpgd, steps_output_gd=steps_output_gd,
                                       steps_objective_gd=steps_objective_gd, seed=s)
                S = summarize(R, X_train, y_train, lam=lam)
                rec = {"epsilon": eps, "lambda": lam, "seed": s}
                rec.update(S[mech])
                records.append(rec)
    return pd.DataFrame(records)


def sweep_dimension(mech, d_list, n_fixed, eps_list, delta, lam, seeds=None,
                    steps_dpgd=1500, steps_output_gd=1500, steps_objective_gd=1500, seed_base=100):
    """
    Vary dimension using synthetic data (so we can control d), keep n fixed.
    Returns long-form DataFrame: d, epsilon, excess_risk, seed.
    """
    records = []
    if seeds is None:
        seeds = list(range(5))
    for d in d_list:
        # regenerate synthetic data at each d; keep n fixed
        Xtr, ytr, Xte, yte = load_dataset(which="synthetic", n=n_fixed, d=d, seed=seed_base)
        for eps in eps_list:
            for s in seeds:
                R = run_all_mechanisms(Xtr, ytr, Xte, yte, lam=lam, eps=eps, delta=delta,
                                       steps_dpgd=steps_dpgd, steps_output_gd=steps_output_gd,
                                       steps_objective_gd=steps_objective_gd, seed=s)
                S = summarize(R, Xtr, ytr, lam=lam)
                rec = {"d": d, "epsilon": eps, "seed": s}
                rec.update(S[mech])
                records.append(rec)
    return pd.DataFrame(records)


def find_min_n_for_target_error(mech, d, eps, delta, lam, target_excess, n_grid, seeds=None,
                                steps_dpgd=1500, steps_output_gd=1500, steps_objective_gd=1500, seed_base=200):
    """
    For fixed (d, eps, delta, lam), find the smallest n in n_grid whose mean excess risk (over seeds)
    is <= target_excess (within a 5% tolerance). Returns (n_found, agg_df).
    """
    if seeds is None:
        seeds = list(range(5))
    best_n = None
    all_rows = []
    for n in sorted(n_grid):
        Xtr, ytr, Xte, yte = load_dataset(which="synthetic", n=n, d=d, seed=seed_base + n)
        recs = []
        for s in seeds:
            R = run_all_mechanisms(Xtr, ytr, Xte, yte, lam=lam, eps=eps, delta=delta,
                                   steps_dpgd=steps_dpgd, steps_output_gd=steps_output_gd,
                                   steps_objective_gd=steps_objective_gd, seed=s)
            S = summarize(R, Xtr, ytr, lam=lam)
            rec = {"n": n, "d": d, "epsilon": eps, "seed": s}
            rec.update(S[mech])
            recs.append(rec)
        df = pd.DataFrame(recs)
        agg = aggregate_seed_results(df, ["n"], "excess_risk")
        agg["d"] = d
        all_rows.append(agg)
        mean_excess = float(agg["excess_risk_mean"].iloc[0])
        if mean_excess <= 1.05 * target_excess:  # 5% tolerance
            best_n = n
            break
    all_df = pd.concat(all_rows, ignore_index=True)
    return best_n, all_df


## Plot (Below is an example code to plot epsilon vs excess risk)

Keep this style of plots.

In [27]:
# =================== Figure generation and saving ===================

def make_fig_q5_lambda_plots():
    seeds = list(range(5))
    delta = 1e-5
    Xtr, ytr, Xte, yte = load_dataset(which="breast_cancer", d=20, seed=22)

    # Small lambdas are fine for output/grad, but NOT for objective (needs λ>c≈0.25)
    lam_list_all = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
    lam_list_obj = [0.3, 0.5, 1.0, 3.0, 10.0]   # enforce λ > 1/4

    eps_list = [0.5, 2.0]  # two epsilons per requirement

    for mech in ["output", "objective", "grad"]:
        lam_list = lam_list_obj if mech == "objective" else lam_list_all
        df = sweep_lambda_for_eps(
            Xtr, ytr, Xte, yte,
            lam_list, eps_list, delta,
            mech=mech, seeds=seeds
        )
        agg = aggregate_seed_results(df, ["epsilon","lambda"], "excess_risk")
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6.2,4.2))
        for (eps_val), sub in agg.groupby("epsilon"):
            sub = sub.sort_values("lambda")
            plt.semilogx(sub["lambda"], sub["excess_risk_mean"], marker='o', linewidth=3, label=f"epsilon={eps_val}")
            plt.fill_between(sub["lambda"], sub["excess_risk_lo"], sub["excess_risk_hi"], alpha=0.12)
        plt.xlabel("Strong convexity λ (log scale)")
        plt.ylabel("Excess empirical risk")
        plt.title(f"{mech.capitalize()} perturbation")
        plt.legend()
        plt.tight_layout()
        fname = f"fig_q5_lambda_{mech}.png"
        plt.savefig(fname, dpi=180)
        plt.close()
        print("Saved:", fname)


def make_fig_q5_dimension_plots():
    seeds = list(range(5))
    delta = 1e-5
    n_fixed = 1000
    d_list = [10, 20, 40, 80, 160, 320]
    eps_list = [0.5, 2.0]

    # Use λ>1/4 for objective; keep your original λ for others
    lam_map = {"output": 1e-2, "grad": 1e-2, "objective": 0.3}

    for mech in ["output", "objective", "grad"]:
        lam_used = lam_map[mech]
        df = sweep_dimension(mech, d_list, n_fixed, eps_list, delta, lam_used, seeds=seeds)
        agg = aggregate_seed_results(df, ["epsilon","d"], "excess_risk")
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6.2,4.2))
        for (eps_val), sub in agg.groupby("epsilon"):
            sub = sub.sort_values("d")
            plt.plot(sub["d"], sub["excess_risk_mean"], marker='o', linewidth=3, label=f"epsilon={eps_val}")
            plt.fill_between(sub["d"], sub["excess_risk_lo"], sub["excess_risk_hi"], alpha=0.12)
        plt.xlabel("Dimension d")
        plt.ylabel("Excess empirical risk")
        plt.title(f"Dimension effect: {mech.capitalize()}")
        plt.legend()
        plt.tight_layout()
        fname = f"fig_q5_dim_{mech}.png"
        plt.savefig(fname, dpi=180)
        plt.close()
        print("Saved:", fname)

def make_fig_q5_scaling_plot():
    mech_list = ["output", "objective", "grad"]
    eps = 1.0
    delta = 1e-5

    # Use λ>1/4 for objective to satisfy the theorem
    lam_map = {"output": 1e-2, "objective": 0.3, "grad": 1e-2}

    d0, n0 = 40, 1000
    seeds = list(range(5))

    # Baseline target excess per mechanism at (d0, n0)
    target = {}
    Xb_tr, yb_tr, Xb_te, yb_te = load_dataset(which="synthetic", n=n0, d=d0, seed=12345)
    for mech in mech_list:
        lam_used = lam_map[mech]
        recs = []
        for s in seeds:
            R = run_all_mechanisms(
                Xb_tr, yb_tr, Xb_te, yb_te,
                lam=lam_used, eps=eps, delta=delta,
                steps_dpgd=1500, steps_output_gd=1500, steps_objective_gd=1500, seed=s
            )
            S = summarize(R, Xb_tr, yb_tr, lam=lam_used)
            recs.append(S[mech]["excess_risk"])
        target[mech] = float(np.mean(recs))

    d_list = [10, 20, 40, 80, 160, 320]
    n_grid = [200, 300, 500, 750, 1000, 1500, 2000, 3000, 4000, 6000, 8000, 12000]

    results = []
    for mech in mech_list:
        lam_used = lam_map[mech]
        for d in d_list:
            n_found, _ = find_min_n_for_target_error(
                mech, d, eps, delta, lam_used,
                target_excess=target[mech],
                n_grid=n_grid, seeds=seeds,
                steps_dpgd=1200, steps_output_gd=1200, steps_objective_gd=1200
            )
            results.append({"mech": mech, "d": d, "n_needed": n_found})

    df_needed = pd.DataFrame(results)

    # Overlay theory curves fitted through (d0, n0)
    d_arr = np.array(d_list, dtype=float)
    c_grad = n0 / np.sqrt(d0)
    c_lin  = n0 / d0

    import matplotlib.pyplot as plt
    plt.figure(figsize=(7.2,4.6))
    for mech in mech_list:
        sub = df_needed[df_needed["mech"]==mech].sort_values("d")
        plt.plot(sub["d"], sub["n_needed"], marker='o', linewidth=3, label=f"{mech}")
    plt.plot(d_arr, c_grad * np.sqrt(d_arr), linestyle='--', linewidth=2, label="theory: c·√d (DP-GD)")
    plt.plot(d_arr, c_lin * d_arr, linestyle='--', linewidth=2, label="theory: c·d (Output/Objective)")
    plt.xlabel("Dimension d")
    plt.ylabel("Minimal n to match baseline excess risk")
    plt.title("Scaling of n with d at fixed (ε,δ) and target excess")
    plt.legend()
    plt.tight_layout()
    fname = "fig_q5_scaling_n_vs_d_all.png"
    plt.savefig(fname, dpi=180)
    plt.close()
    print("Saved:", fname)


# -------- Run all figure generators (you can comment/uncomment as needed) --------
make_fig_q5_lambda_plots()
make_fig_q5_dimension_plots()
make_fig_q5_scaling_plot()


Saved: fig_q5_lambda_output.png
Saved: fig_q5_lambda_objective.png
Saved: fig_q5_lambda_grad.png
Saved: fig_q5_dim_output.png
Saved: fig_q5_dim_objective.png
Saved: fig_q5_dim_grad.png
Saved: fig_q5_scaling_n_vs_d_all.png
